# 02 — Full Export Pipeline

**Goal:** Kick off GEE export tasks for **all 12 layers** needed for the cube:

| # | Layer | Period | Bands | File |
|---|-------|--------|-------|------|
| 1 | CHIRPS rain (mm/month) | 2003–2026 | 282 | `rain_mm_monthly_2003_2026.tif` |
| 2 | ERA5 2m temperature (K) | 2003–2026 | 282 | `t2m_k_monthly_2003_2026.tif` |
| 3 | MODIS NDVI | 2003–2026 | 282 | `ndvi_monthly_2003_2026.tif` |
| 4 | MODIS EVI | 2003–2026 | 282 | `evi_monthly_2003_2026.tif` |
| 5 | MODIS ET (mm/day) | 2003–2026 | 282 | `et_monthly_2003_2026.tif` |
| 6 | MODIS LST daytime (raw DN) | 2003–2026 | 282 | `lst_day_k_monthly_2003_2026.tif` |
| 7 | SMAP soil moisture (m³/m³) | 2015–2026 | 138 | `sm_surf_monthly_2015_2026.tif` |
| 8 | ERA5 dewpoint (K) | 2003–2026 | 282 | `dewpoint_k_monthly_2003_2026.tif` |
| 9 | ERA5 wind speed (m/s) | 2003–2026 | 282 | `wind_speed_monthly_2003_2026.tif` |
| 10 | Elevation (m) + Slope (°) | static | 2 | `static_elev_slope.tif` |
| 11 | ESA WorldCover land cover | static | 1 | `static_landcover.tif` |
| 12 | Land mask | static | 1 | `land_mask.tif` |

> **Note on SMAP:** SMAP data starts ~March 2015 in GEE. We export from 2015-01 (Jan–Feb will be NaN-masked). The 2003–2014 gap is also NaN-masked in cube assembly.

> **Note on data lag:** The most recent 1–2 months of each variable may be partially masked — CHIRPS has ~3 week lag, ERA5-Land ~6 weeks. GEE fills missing months automatically so band counts stay exact.

> **Note on scale factors:** Several MODIS products need scaling when loaded in Python:
> - `ndvi`, `evi`: multiply × 0.0001
> - `lst_day_k`: multiply × 0.02 (Kelvin), then −273.15 for °C
> - `et`: multiply × 0.1 (mm/day per 8-day composite)
> - `t2m_k`, `dewpoint_k`: already Kelvin — subtract 273.15 for °C

Run **top to bottom** once. Tasks are async — GEE runs them in the background.
Check progress at https://code.earthengine.google.com → **Tasks tab**.
Expect 10–30 minutes per variable; all run in parallel.

In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run ONLY on Google Colab. Skip entirely if running locally.
try:
    from google.colab import drive
    import subprocess, sys
    drive.mount('/content/drive')
    _REPO = '/content/drive/MyDrive/botswana-drought-flood'
    _r = subprocess.run(['git', 'pull'], cwd=_REPO, capture_output=True, text=True)
    print(_r.stdout.strip() or 'Already up to date.')
    if _r.returncode != 0:
        print('git pull warning:', _r.stderr.strip())
    sys.path.insert(0, f'{_REPO}/src')
    print(f'src → {_REPO}/src')
except ImportError:
    pass  # running locally — botswana_ds already on sys.path

In [1]:
import ee, geemap
from ee import oauth as _oauth

_flow = _oauth.Flow('notebook')

print("Step 1 — open this URL in your browser and approve access:")
print()
print(_flow.auth_url)
print()

_code = input("Step 2 — paste the authorization code here: ")

ee.Authenticate(authorization_code=_code.strip(), code_verifier=_flow.code_verifier)
ee.Initialize(project='infosys-drought-prediction')
print('Earth Engine ready.')

Step 1 — open this URL in your browser and approve access:

https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=MldrWzKxqZOEHsGaw0i2DrsgufnfGArPQ514QbofwN0&tc=hYvDPjp3aSTUGKi__UCJBXog_0iByMLxhkRED9y_jD0&cc=HCxoo6GOua3dYIJq56bHWGJfBXeDntAAmxQ1vdtBWug


Successfully saved authorization token.
Earth Engine ready.


---
## Step 1 — Dynamic layers (8 variables via DROUGHT_CORE)

These 8 variables share the same export function. We loop over them.
Each kicks off one GEE batch task and returns immediately — they run in parallel on GEE's servers.

In [ ]:
import ee
ee.Initialize(project='infosys-drought-prediction')

from botswana_ds.gee.export import export_variable_to_drive, DROUGHT_CORE

START  = '2003-01-01'
END    = '2026-07-01'   # exclusive — covers Jan 2003 through Jun 2026 = 282 months
FOLDER = 'BotswanaDroughtFloodSet'

# sm_surf is exported separately (SMAP only goes back to 2015, not 2003)
SKIP = {'sm_surf'}

tasks = {}
for name in DROUGHT_CORE:
    if name in SKIP:
        print(f'  {name:12s}  skipped (exported separately below)')
        continue
    t = export_variable_to_drive(name, start=START, end=END, folder=FOLDER)
    tasks[name] = t
    print(f'  {name:12s}  task_id={t.id}')

print(f'\n{len(tasks)} export tasks started.')
print('Monitor at https://code.earthengine.google.com → Tasks')

---
## Step 2 — SMAP soil moisture (shorter time range)

SMAP was launched in 2015. The GEE collection `NASA/SMAP/SPL4SMGP/008` starts ~March 2015.
We export from 2015-01 (Jan–Feb will be fully masked). From Jan 2015 to Jun 2026 = 138 months.
The 2003–2014 gap is NaN-masked in cube assembly.

In [ ]:
import ee
ee.Initialize(project='infosys-drought-prediction')

from botswana_ds.gee.export import export_variable_to_drive

t_smap = export_variable_to_drive(
    'sm_surf',
    start='2015-01-01',
    end='2026-07-01',
    folder='BotswanaDroughtFloodSet',
)
print(f'SMAP export started — task_id={t_smap.id}')
print('Covers Jan 2015 – Jun 2026 = 138 months (Jan–Feb 2015 will be NaN-masked).')

---
## Step 3 — Wind speed (needs special handling)

ERA5-Land stores wind as two components (u, v). Wind speed = √(u² + v²).
This is the one channel that doesn't fit the single-band `DROUGHT_CORE` pattern,
so it has its own export function.

In [ ]:
import ee
ee.Initialize(project='infosys-drought-prediction')

from botswana_ds.gee.export import export_wind_speed_to_drive

t_wind = export_wind_speed_to_drive(
    start='2003-01-01',
    end='2026-07-01',
    folder='BotswanaDroughtFloodSet',
)
print(f'Wind speed export started — task_id={t_wind.id}')

---
## Step 4 — Static layers (elevation, slope, land cover, land mask)

These don't change over time. We export them once and reuse across all time steps.

- **Elevation + slope**: Copernicus GLO-30 DEM (30 m → resampled to 0.05°). Two bands in one file.
- **Land cover**: ESA WorldCover 2021 (10 m → resampled). Integer class codes (10=Trees, 20=Shrubs, …).
- **Land mask**: 1 inside Botswana, 0 outside. Used to exclude ocean/border pixels from the loss.

In [5]:
import ee
ee.Initialize(project='infosys-drought-prediction')

from botswana_ds.gee.export import export_static_layers, export_land_mask

static_tasks = export_static_layers(folder='BotswanaDroughtFloodSet')
for t in static_tasks:
    print(f'  {t.config["description"]:25s}  task_id={t.id}')

t_mask = export_land_mask(folder='BotswanaDroughtFloodSet')
print(f'  {"land_mask":25s}  task_id={t_mask.id}')

print(f'\n{len(static_tasks) + 1} static export tasks started.')

  static_elev_slope          task_id=ILBWZUWNMA4QXOHZ2LFQEVCE
  static_landcover           task_id=CQG6VQDXH64A3OXJD6ULVZY4
  land_mask                  task_id=A35J7OYOYTNFHKDP7OLTIRJI

3 static export tasks started.


---
## Step 5 — Poll task status

Re-run this cell anytime to see which tasks have finished.
You don't need to wait — just come back in 30–60 minutes.

In [6]:
import ee
ee.Initialize(project='infosys-drought-prediction')

all_tasks = list(tasks.values()) + [t_smap, t_wind] + static_tasks + [t_mask]

print(f'{"Task":35s}  {"Status":12s}  {"Error"}')
print('-' * 70)
for t in all_tasks:
    status = t.status()
    state  = status['state']
    desc   = status.get('description', '?')
    error  = status.get('error_message', '')
    print(f'{desc:35s}  {state:12s}  {error}')

Task                                 Status        Error
----------------------------------------------------------------------
rain_mm_2003_2024                    RUNNING       
t2m_k_2003_2024                      RUNNING       
ndvi_2003_2024                       READY         
evi_2003_2024                        READY         
et_2003_2024                         READY         
lst_day_k_2003_2024                  READY         
dewpoint_k_2003_2024                 READY         
sm_surf_2015_2024                    READY         
wind_speed_2003_2024                 READY         
static_elev_slope                    READY         
static_landcover                     READY         
land_mask                            READY         


---
## Step 6 — Verify one completed file

Once **at least one** task shows `COMPLETED`, download that file from
`My Drive/BotswanaDroughtFloodSet/` to `data/` and run this cell to confirm shape.

Expected for a 2003–2026 variable: shape `(282, 182, 188)` — 282 months, 182 rows, 188 cols.

In [ ]:
import rioxarray as rxr
import numpy as np
from pathlib import Path
from botswana_ds.grid import make_grid

# Change this to whichever file finished first
FILENAME = 'rain_mm_monthly_2003_2026.tif'
EXPECTED_BANDS = 282   # Jan 2003 – Jun 2026

tif = Path('..') / 'data' / FILENAME
if not tif.exists():
    raise FileNotFoundError(
        f'Not found: {tif}\n'
        f'Download {FILENAME} from Google Drive → BotswanaDroughtFloodSet/ '
        f'and save it to data/'
    )

da = rxr.open_rasterio(tif, masked=True)
g  = make_grid()
H, W = g.shape

print(f'File              : {FILENAME}')
print(f'Shape (B, H, W)   : {da.shape}')
print(f'CRS               : {da.rio.crs}')
print(f'Resolution        : {abs(float(da.rio.resolution()[0])):.5f}°')

bands_ok = da.shape[0] == EXPECTED_BANDS
shape_ok = da.shape[1] == H and da.shape[2] == W
print(f'\nBands match {EXPECTED_BANDS}    : {"PASS" if bands_ok else f"FAIL — got {da.shape[0]}"}')
print(f'Grid match {H}×{W}  : {"PASS" if shape_ok else f"FAIL — got {da.shape[1]}×{da.shape[2]}"}')

vals = da.values
valid = vals[~np.isnan(vals)]
print(f'\nValue range (all bands): {valid.min():.2f} – {valid.max():.2f}')
print(f'NaN fraction           : {np.isnan(vals).mean():.1%}')
# Higher NaN% in recent months is expected (data lag)
print('Note: the most recent 1–2 months may be fully NaN due to processing lag.')

verification:

In [ ]:
import rioxarray as rxr
from pathlib import Path
from botswana_ds.grid import make_grid

DATA = Path('..') / 'data'
H, W = make_grid().shape  # 182 × 188

EXPECTED = {
    'rain_mm_monthly_2003_2026.tif':      282,
    't2m_k_monthly_2003_2026.tif':        282,
    'ndvi_monthly_2003_2026.tif':         282,
    'evi_monthly_2003_2026.tif':          282,
    'et_monthly_2003_2026.tif':           282,
    'lst_day_k_monthly_2003_2026.tif':    282,
    'sm_surf_monthly_2015_2026.tif':      138,
    'dewpoint_k_monthly_2003_2026.tif':   282,
    'wind_speed_monthly_2003_2026.tif':   282,
    'static_elev_slope.tif':                2,
    'static_landcover.tif':                 1,
    'land_mask.tif':                        1,
}

print(f'{"File":42s}  {"Expected":8s}  {"Got":10s}  Status')
print('-' * 75)
all_pass = True
for fname, exp_bands in EXPECTED.items():
    path = DATA / fname
    if not path.exists():
        print(f'{fname:42s}  {exp_bands:8d}  {"MISSING":10s}  ✗')
        all_pass = False
        continue
    da = rxr.open_rasterio(path, masked=True)
    b, h, w = da.shape
    ok = (b == exp_bands) and (h == H) and (w == W)
    status = 'PASS' if ok else f'FAIL (got {b}×{h}×{w})'
    print(f'{fname:42s}  {exp_bands:8d}  {str(b):10s}  {status}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('✓ All files verified. Ready for NB03 cube assembly.')
else:
    print('✗ Some files missing or wrong shape. Check GEE Tasks tab.')

In [ ]:
import ee
ee.Initialize(project='infosys-drought-prediction')

from botswana_ds.gee.export import export_variable_to_drive

# Re-run ET export if the main loop failed for it (MODIS/006 deprecation warning is harmless)
t_et = export_variable_to_drive('et', start='2003-01-01', end='2026-07-01',
                                  folder='BotswanaDroughtFloodSet')
print(f'ET  task_id={t_et.id}')

---
## Success criteria

| Check | Expected |
|-------|----------|
| All 12 tasks started | 12 task IDs printed |
| All tasks COMPLETED (check Tasks tab) | No FAILED tasks |
| Shape of a dynamic file | `(282, 182, 188)` — Jan 2003 – Jun 2026 |
| Shape of SMAP file | `(138, 182, 188)` — Jan 2015 – Jun 2026 |
| Shape of static file | `(2, 182, 188)` (elev+slope) or `(1, 182, 188)` |
| Values physically reasonable | Rain ≥ 0, LST DN 10000–20000, NDVI DN −2000–10000 |
| Last 1–2 months fully NaN | Expected — data lag is normal |

**All boxes ticked?** Download the new `.tif` files from Google Drive → `data/`, then re-run:

1. **NB03** — re-assemble the cube with the extended time range
2. **NB05** — re-run baselines (XGBoost) to generate predictions for the new months
3. **`scripts/export_app_data.py`** — refresh `data/app/` so the Streamlit app picks up the new dates

Files to have in `data/` before starting NB03:
```
rain_mm_monthly_2003_2026.tif
t2m_k_monthly_2003_2026.tif
ndvi_monthly_2003_2026.tif
evi_monthly_2003_2026.tif
et_monthly_2003_2026.tif
lst_day_k_monthly_2003_2026.tif
sm_surf_monthly_2015_2026.tif
dewpoint_k_monthly_2003_2026.tif
wind_speed_monthly_2003_2026.tif
static_elev_slope.tif
static_landcover.tif
land_mask.tif
```